# H4. A hidden layer changes the representation
Book: Neural Networks, Classification with Neural Networks.

Supplement: Alisa's book of LLMs, [Multi-layer perceptrons](https://alisawuffles.notion.site/alisa-s-book-of-llms#3157eb87360580ffbe0fed2fb99db7e1) and [Activation functions](https://alisawuffles.notion.site/alisa-s-book-of-llms#2d17eb873605805194b1dfc597abdcd4). Focus on batch shapes, sigmoid, and ReLU.
These selected readings are optional support. The classroom examples define the required scope.
Reading caution: ReLU's range includes zero. Sigmoid can occur in hidden
computations, although our classroom network uses ReLU hidden units.
Computing connection: [Alisa's FLOP accounting](https://alisawuffles.notion.site/alisa-s-book-of-llms#3027eb873605803e9fdfff2be7333715).
Use the dense-product counting idea only. Transformer formulas are later material.

Reuse the circle data and split from H1, using the book's radial ranges scaled by 10.

## Setup
Run this cell once. Helpers support the experiments below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (8, 4.5), 'font.size': 12})

def circle_data(seed=601, n=200):
    rng = np.random.default_rng(seed)
    n0 = n // 2
    radius = np.r_[rng.uniform(0, .4, n0), rng.uniform(.8, 1, n-n0)]
    angle = rng.uniform(0, 2*np.pi, n)
    x = np.c_[radius*np.cos(angle), radius*np.sin(angle)]
    y = np.r_[np.zeros(n0), np.ones(n-n0)].astype(int)
    return x, y

def circle_split():
    from sklearn.model_selection import train_test_split
    x, y = circle_data()
    xa, xt, ya, yt = train_test_split(x, y, test_size=.2, stratify=y,
                                     random_state=600)
    xr, xv, yr, yv = train_test_split(xa, ya, test_size=.25, stratify=ya,
                                     random_state=600)
    return xr, xv, xt, yr, yv, yt

def radial_features(x):
    return np.sum(x*x, axis=1, keepdims=True)

def logistic_model(radial=False):
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler, FunctionTransformer
    from sklearn.linear_model import LogisticRegression
    steps = [FunctionTransformer(radial_features)] if radial else []
    return make_pipeline(*steps, StandardScaler(), LogisticRegression(C=1.0))

def classifier_plot(probability, x, y, ax=None, title=""):
    if ax is None:
        _, ax = plt.subplots()
    g = np.linspace(-1.1, 1.1, 130)
    xx, yy = np.meshgrid(g, g)
    p = probability(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, p, levels=np.linspace(0, 1, 11), cmap="RdBu_r",
                alpha=.35, vmin=0, vmax=1)
    if p.min() < .5 < p.max():
        ax.contour(xx, yy, p, levels=[.5], colors="black", linewidths=1.5)
    ax.scatter(x[:, 0], x[:, 1], c=y, cmap="RdBu_r", vmin=0, vmax=1,
               edgecolors="white", s=25)
    ax.set(xlabel="x1", ylabel="x2", title=title, aspect="equal")
    return ax

def train_network(xr, yr, xv, yv, lr=.5, epochs=600, width=4,
                  seed=1, update=True, batch_size=None):
    import torch
    torch.set_num_threads(1)
    torch.manual_seed(seed)
    model = torch.nn.Sequential(torch.nn.Linear(2, width), torch.nn.ReLU(),
                                torch.nn.Linear(width, 1))
    x = torch.tensor(xr, dtype=torch.float32)
    y = torch.tensor(yr[:, None], dtype=torch.float32)
    vx = torch.tensor(xv, dtype=torch.float32)
    vy = torch.tensor(yv[:, None], dtype=torch.float32)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = {"train": [], "validation": []}
    batch_size = len(y) if batch_size is None else batch_size
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(y))
        for start in range(0, len(y), batch_size):
            idx = order[start:start+batch_size]
            optimizer.zero_grad()
            loss = loss_fn(model(x[idx]), y[idx])
            loss.backward()
            if update:
                optimizer.step()
        model.eval()
        with torch.no_grad():
            history["train"].append(loss_fn(model(x), y).item())
            history["validation"].append(loss_fn(model(vx), vy).item())
        if not np.isfinite(history["train"][-1]):
            break
    return model, history

def network_probability(model, x):
    import torch
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(x, dtype=torch.float32))).numpy().ravel()

def dense_cost(batch, inputs, outputs, dtype="float32"):
    """Approximate dense-matmul FLOPs and resident element bytes, without allocation."""
    for value in (batch, inputs, outputs):
        if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < 1:
            raise ValueError("Dimensions must be positive integers.")
    data_type = np.dtype(dtype)
    if data_type not in (np.dtype("float32"), np.dtype("float64")):
        raise ValueError("Use float32 or float64 for this experiment.")
    batch, inputs, outputs = int(batch), int(inputs), int(outputs)
    size = data_type.itemsize
    return {"flops": 2*batch*inputs*outputs, "input_bytes": size*batch*inputs,
            "weight_bytes": size*inputs*outputs, "output_bytes": size*batch*outputs}

def benchmark_dense(batch=64, inputs=64, outputs=64, repeats=7, dtype="float32", seed=600):
    """Compare warmed CPU row and batch products, including their call overhead."""
    from time import perf_counter
    cost = dense_cost(batch, inputs, outputs, dtype)
    if isinstance(repeats, bool) or not isinstance(repeats, (int, np.integer)) or repeats < 1:
        raise ValueError("Repeats must be a positive integer.")
    rng = np.random.default_rng(seed)
    x = rng.normal(size=(batch, inputs)).astype(dtype)
    w = rng.normal(size=(inputs, outputs)).astype(dtype)
    rows = np.empty((batch, outputs), dtype=dtype)
    together = np.empty_like(rows)

    def row_product():
        for i in range(batch):
            np.matmul(x[i], w, out=rows[i])

    def batch_product():
        np.matmul(x, w, out=together)

    operations = {"row": row_product, "batch": batch_product}
    for operation in operations.values():
        for _ in range(3):
            operation()
    samples = {name: [] for name in operations}
    calls_per_block = 20
    for repetition in range(repeats):
        order = ["row", "batch"] if repetition % 2 == 0 else ["batch", "row"]
        for name in order:
            start = perf_counter()
            for _ in range(calls_per_block):
                operations[name]()
            samples[name].append((perf_counter()-start)*1000/calls_per_block)
    tolerance = 1e-4 if np.dtype(dtype) == np.dtype("float32") else 1e-10
    np.testing.assert_allclose(rows, together, rtol=tolerance, atol=tolerance)
    return {"row_median_ms": float(np.median(samples["row"])),
            "batch_median_ms": float(np.median(samples["batch"])),
            "max_abs_error": float(np.max(np.abs(rows-together))), "cost": cost}

## A. A useful feature (30 minutes)
Predict the effect of replacing the raw coordinates with x1 squared + x2 squared.
Both logistic fits use the same fixed penalty. This avoids infinite coefficients
on the separable radial example.

In [ ]:
from sklearn.metrics import log_loss
xr, xv, xt, yr, yv, yt = circle_split()
raw = logistic_model().fit(xr, yr)
radial = logistic_model(radial=True).fit(xr, yr)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, model, name in zip(axes, [raw, radial], ["Raw coordinates", "Radius squared"]):
    classifier_plot(lambda x: model.predict_proba(x)[:, 1], xr, yr, ax=ax, title=name)
    print(name, "validation log loss", log_loss(yv, model.predict_proba(xv)))
plt.tight_layout()
plt.show()

Describe what changed in the representation and what stayed the same in the loss.

Prediction:

Observation:

Explanation:

## B. Forward pass (10 minutes), shape/cost/timing (8), and bias change (12)
Calculate this example on paper. Inputs are rows. Predict the shapes before running.

In [ ]:
X = np.array([[1., -1.]])
W1 = np.array([[1., -1.], [.5, 1.]])
b1 = np.array([0., .5])
W2 = np.array([[2.], [-1.]])
b2 = np.array([-.5])
H = np.maximum(0, X @ W1 + b1)
z = H @ W2 + b2
print("Hidden features:", H, "logit:", z, "probability:", 1/(1+np.exp(-z)))
assert H.shape == (1, 2) and z.shape == (1, 1)

The next cell supplies a fitted network so we can inspect its representation.
Training is the next meeting's topic. Predict the effect of changing one hidden bias.

In [ ]:
import torch
network, history = train_network(xr, yr, xv, yv)
classifier_plot(lambda x: network_probability(network, x), xr, yr,
                title="Four learned ReLU features")
plt.show()
print("Number of parameters:", sum(p.numel() for p in network.parameters()))
assert sum(p.numel() for p in network.parameters()) == 17

Within B, predict the batch and stored weight shapes before running this check.
The board uses input-by-output weights; PyTorch stores output-by-input weights.

In [ ]:
batch = torch.tensor(xr[:20], dtype=torch.float32)
with torch.no_grad():
    hidden = network[1](network[0](batch))
    logits = network(batch)
print("Input:", tuple(batch.shape), "hidden:", tuple(hidden.shape),
      "logits:", tuple(logits.shape))
print("Stored weights:", tuple(network[0].weight.shape), tuple(network[2].weight.shape))
assert hidden.shape == (20, 4) and logits.shape == (20, 1)

### Cost and timing, within the eight-minute shape/computing block
Predict what doubles when batch size doubles: FLOPs, output bytes, weight bytes?
First estimate the larger layer on paper. The cost helper does not allocate it.

In [ ]:
example_cost = dense_cost(64, 1024, 1024)
print("Estimated FLOPs:", example_cost["flops"])
for name in ["input_bytes", "weight_bytes", "output_bytes"]:
    print(name, example_cost[name], "or", example_cost[name]/2**20, "MiB")

Now time a smaller isolated matrix product, not the fitted classification model.
Both implementations use preallocated outputs. The row version runs `x[i] @ w`
inside a Python loop; the batch version runs `x @ w` once.
Data generation is outside timing. Both implementations warm up and repeat.
Predict which will be faster, then explain what your machine actually does.

In [ ]:
import platform
print("CPU platform:", platform.machine(), "NumPy:", np.__version__)
for batch_size in [64, 128]:
    result = benchmark_dense(batch=batch_size, inputs=64, outputs=64)
    print("Batch", batch_size, "FLOPs", result["cost"]["flops"],
          "row ms", round(result["row_median_ms"], 5),
          "batch ms", round(result["batch_median_ms"], 5),
          "maximum output difference", result["max_abs_error"])

Timings depend on the CPU, numerical library, array layout, and thread settings.
Run `np.show_config()` if comparing library builds. Repeat the measurement.
Do not claim a universal speedup or infer GPU performance from this CPU example.
The helper checks approximate output agreement, not identical final bits.

Prediction:

Observation:

Explanation:

Optional precision check: predict the effect of adding 1 to 100,000,000.
Float64 doubles element storage relative to float32. Precision and speed are
separate questions; neither dtype is always the right choice.

In [ ]:
for dtype in [np.float32, np.float64]:
    print(dtype.__name__, "bytes/element", np.dtype(dtype).itemsize,
          "increment retained", (dtype(1e8)+dtype(1))-dtype(1e8))

### One hidden bias (12 minutes, finishing B)
Predict the change, run once, and explain its mechanism before trying variants.

In [ ]:
import copy
changed = copy.deepcopy(network)
hidden_unit = 0
bias_change = .5
with torch.no_grad():
    changed[0].bias[hidden_unit] += bias_change
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
classifier_plot(lambda x: network_probability(network, x), xr, yr, axes[0], "Original")
classifier_plot(lambda x: network_probability(changed, x), xr, yr, axes[1], "One bias changed")
plt.tight_layout()
plt.show()

Explain the changed boundary in terms of the hidden unit. Do not retrain yet.

Prediction:

Observation:

Explanation:

## Individual exit
Label the dimensions in a 2-input, 4-hidden-unit, 1-output network.
Why do affine layers without nonlinear activations collapse to one affine map?
If batch size doubles, what happens to work, output storage, and parameter storage?